# Star-Diff + PixelGen: Dual-Path Restoration Diffusion for H&E → IHC
## Using PixelGen JiT_I2I XL Backbone (Pretrained)

Combines:
- **Star-Diff** (arXiv:2508.02528): Dual-path restoration diffusion
- **PixelGen** (arXiv:2602.02493): JiT_I2I ViT backbone + perceptual losses

Architecture: Two independent **JiT_I2I_XL** backbones:
- **Noise path** (ε_θ): predicts Gaussian noise ε
- **Restoration path** (r_θ): predicts I_res = I_ihc - I_he


In [1]:
# Install dependencies
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130
%pip install -q accelerate wandb kaggle lpips einops tqdm
%pip install -q "diffusers[torch]" transformers huggingface_hub
%pip install -q scikit-image pillow matplotlib opencv-python scipy
%pip install -q pytorch-msssim


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## 1. Environment Setup & Download Weights

In [2]:
%load_ext autoreload
%autoreload 2

%load_ext autoreload
%autoreload 2

import sys, os

# Ensure PixelGen root is on path
PIXELGEN_ROOT = os.path.abspath(".")
if PIXELGEN_ROOT not in sys.path:
    sys.path.insert(0, PIXELGEN_ROOT)

# Download the pretrained XL 80ep (FMonly) weights directly
weight_url = "https://huggingface.co/zehongma/PixelGen/resolve/main/PixelGen_XL_80ep.ckpt"
weight_path = os.path.join(PIXELGEN_ROOT, "PixelGen_XL_80ep.ckpt")

if not os.path.exists(weight_path):
    print(f"Downloading weights to {weight_path}...")
    os.system(f"wget -q {weight_url} -O {weight_path}")
else:
    print(f"✓ Weights already downloaded: {weight_path}")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✓ Weights already downloaded: /teamspace/studios/this_studio/PixelGen/PixelGen_XL_80ep.ckpt


## 2. API Keys & W&B Login

In [3]:
import os
import wandb
from huggingface_hub import login as hf_login

# Set your keys here
os.environ["HF_TOKEN"] = ""
os.environ["WANDB_API_KEY"] = ""
os.environ["KAGGLE_USERNAME"] = "ahmedayman4a77"

hf_login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
wandb.login()
print(f"✓ W&B: {wandb.Api().viewer.username}")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: ahmedayman4a77 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✓ W&B: ahmedayman4a77


## 3. Configuration (Forced XL Architecture)

We are forcing the **XL** architecture to leverage the pretrained .
Caution: Training two XL models will consume significant VRAM (> 50GB typical with checkpointing).

In [4]:
from stardiff_pixelgen.config import StarDiffPixelGenConfig, detect_gpu

# gpu_type, batch_size, grad_accum, precision = detect_gpu()

config = StarDiffPixelGenConfig()
# config.update_from_gpu(gpu_type, batch_size, grad_accum, precision)

# ── OVERRIDE: Point to the downloaded weights ──
config.pretrained_weight_path = weight_path

effective = config.batch_size * config.gradient_accumulation_steps
print(f"\nConfig: JiT_I2I_{config.model_size} (hidden={config.hidden_size}, depth={config.depth})")
print(f"Batch: {config.batch_size} × {config.gradient_accumulation_steps} = {effective}")
print(f"EMA: decay={config.ema_decay} | LPIPS: {config.lpips_weight} | DINO: {config.use_dino}")



Config: JiT_I2I_XL (hidden=1152, depth=28)
Batch: 128 × 1 = 128
EMA: decay=0.999 | LPIPS: 0 | DINO: False


## 4. Download Dataset & Create DataLoaders

In [5]:
from stardiff_pixelgen.dataset import download_dataset, create_dataloaders

# download_dataset(config.dataset_root, config.stains)
train_loader, val_loader = create_dataloaders(config)


  Loaded 17041 paired H&E/IHC images from /teamspace/studios/this_studio/data/KI67/train/he
🚀 Preloading 17041 images into System RAM (as uint8) to eliminate IO bottleneck...


Preloading: 100%|██████████| 17041/17041 [03:16<00:00, 86.60it/s]


  Loaded 3900 paired H&E/IHC images from /teamspace/studios/this_studio/data/KI67/val/he
🚀 Preloading 3900 images into System RAM (as uint8) to eliminate IO bottleneck...


Preloading: 100%|██████████| 3900/3900 [00:45<00:00, 86.11it/s]

✓ Train: 17041 samples, 133 batches
✓ Val:   3900 samples, 31 batches


## 5. Create StarDiff Model (Loads Pretrained Weights)

This will initialize both networks from .

In [6]:
import torch
from stardiff_pixelgen.model import create_stardiff_model, SimpleEMA

device = "cuda" if torch.cuda.is_available() else "cpu"
model = create_stardiff_model(config, device)

# EMA tracker
ema_tracker = None
if config.use_ema:
    ema_tracker = SimpleEMA(model, decay=config.ema_decay, every_n_steps=config.ema_every_n_steps)


✓ Gradient checkpointing enabled for both paths
✓ StarDiff-PixelGen: 2 × JiT_I2I backbones
  Noise path:       676.7M params
  Restoration path: 676.7M params
  Total:            1353.4M params
  Hidden: 1152, Depth: 28, Heads: 16
  Bottleneck: False (dim=128)
  Gradient checkpointing: True
  Pretrained from: /teamspace/studios/this_studio/PixelGen/PixelGen_XL_80ep.ckpt

✓ Model on cuda: 1353.4M total, 1352.8M trainable
  Architecture: JiT_I2I_XL


In [7]:
from stardiff_pixelgen.stardiff_scheduler import StarDiffScheduler

scheduler = StarDiffScheduler(
    num_timesteps=config.num_timesteps,
    restoration_weight=config.restoration_weight,
    he_init_alpha=config.he_init_alpha,
)


In [8]:
from stardiff_pixelgen.losses import PixelGenPerceptualLoss

perceptual_loss_fn = PixelGenPerceptualLoss(
    use_dino=config.use_dino,
    lpips_weight=config.lpips_weight,
    dino_weight=config.dino_weight,
    noise_gate_threshold=config.noise_gate_threshold,
    device=device,
)


Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/lpips/weights/v0.1/vgg.pth
✓ LPIPS (VGG) loaded — 14.7M (frozen)


In [9]:
import gc
from torch.optim import AdamW
from accelerate import Accelerator
from diffusers.optimization import get_constant_schedule_with_warmup

gc.collect()
torch.cuda.empty_cache()

try:
    import bitsandbytes as bnb
    optimizer = bnb.optim.AdamW8bit(model.parameters(), lr=config.learning_rate, betas=(0.9, 0.999), weight_decay=1e-2)
except ImportError:
    print('⚠️ bitsandbytes not found. Falling back to standard AdamW.')
    optimizer = AdamW(model.parameters(), lr=config.learning_rate, betas=(0.9, 0.999), weight_decay=1e-2)

lr_scheduler = get_constant_schedule_with_warmup(optimizer, num_warmup_steps=config.warmup_steps)

accelerator = Accelerator(
    mixed_precision=config.mixed_precision,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
)
model, optimizer, train_loader, val_loader, lr_scheduler = accelerator.prepare(
    model, optimizer, train_loader, val_loader, lr_scheduler
)
print(f"✓ Accelerator ready: {config.mixed_precision} precision")


⚠️ bitsandbytes not found. Falling back to standard AdamW.
✓ Accelerator ready: bf16 precision


## 7. Launch Training

## 6. Optional: Resume from W&B Checkpoint

If you have a previous run interrupted, you can download the best or latest checkpoint from W&B and resume from exactly where you left off.

In [10]:
from stardiff_pixelgen.train import resume_from_wandb

# 1. Set the run path and filename
resume_run_id = "dfvt75jy"
run_path = f"mohamed-tarek2607/star-diff-he-to-ihc/{resume_run_id}"
checkpoint_file = "stardiff_epoch_27.pt"

resume_training = False  # Enabled for you
start_epoch = 0

if resume_training and resume_run_id:
    start_epoch = resume_from_wandb(
        run_path=run_path,
        file_name=checkpoint_file,
        model=model, 
        optimizer=optimizer, 
        lr_scheduler=lr_scheduler, 
        accelerator=accelerator,
        ema_tracker=ema_tracker
    )
    print(f"✓ Ready to resume from epoch {start_epoch}")
else:
    print("Starting fresh training from epoch 0")

Starting fresh training from epoch 0


In [ ]:
from stardiff_pixelgen.train import train_stardiff

print("=" * 60)
print("Starting StarDiff + PixelGen Training: H&E → IHC")
print("=" * 60)

trained_model = train_stardiff(
    model=model,
    scheduler=scheduler,
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    optimizer=optimizer,
    lr_scheduler=lr_scheduler,
    accelerator=accelerator,
    config=config,
    perceptual_loss_fn=perceptual_loss_fn,
    ema_tracker=ema_tracker,
    start_epoch=start_epoch,
    he_init_alpha=config.he_init_alpha,
)


Starting StarDiff + PixelGen Training: H&E → IHC


Epoch 1/50:  14%|█▍        | 19/133 [01:09<06:34,  3.46s/it, dab=0.0000, loss=1.1833, noise=0.2413, perc=0.0000, rest=0.9420]

## 8. Comprehensive Evaluation (FID, KID, SSIM, PSNR, MAD)

Evaluate the model on the validation dataset using standard metrics from `src/eval/metrics.py`.

In [ ]:
import torch
import numpy as np
from tqdm import tqdm
from src.eval.metrics import (
    compute_fid, compute_kid, compute_ssim_batch, 
    compute_psnr_batch, compute_mad_batch
)
import lpips
import torchvision.transforms.functional as TF

# 1. Setup evaluator and metrics
device = "cuda" if torch.cuda.is_available() else "cpu"
lpips_fn = lpips.LPIPS(net="alex").to(device)

model.eval()
all_preds = []
all_gts = []

ssim_scores = []
psnr_scores = []
mad_scores = []
lpips_scores = []

def denorm(x):
    return ((x + 1) / 2).clamp(0, 1)

# 2. Generate predictions on validation set
print(f"Generating predictions for {len(val_loader.dataset)} validation samples...")
for batch in tqdm(val_loader, desc="Evaluation"):
    he = batch["he"].to(device)
    ihc = batch["ihc"].to(device)
    
    with torch.no_grad():
        # Use the accelerator's unwrapped model
        unwrapped_model = accelerator.unwrap_model(model)
        shape = (he.shape[0], 3, config.image_size, config.image_size)
        
        gen = scheduler.sample(
            unwrapped_model,
            he, shape, 
            device=device,
            use_restoration=True, use_noise=True,
            progress=False
        )
        
        gen_01 = denorm(gen)
        ihc_01 = denorm(ihc)
        
        # Batch-based metrics
        ssim_scores.extend(compute_ssim_batch(gen_01, ihc_01))
        psnr_scores.extend(compute_psnr_batch(gen_01, ihc_01))
        mad_scores.extend(compute_mad_batch(gen_01, ihc_01))
        
        # LPIPS expects [-1, 1]
        l_scores = lpips_fn(gen.to(device), ihc.to(device)).view(-1).cpu().tolist()
        lpips_scores.extend(l_scores)
        
        # Collect for FID/KID (CPU to save VRAM)
        all_preds.append(gen_01.cpu())
        all_gts.append(ihc_01.cpu())

# 3. Aggregate images for Distributional Metrics (FID/KID)
all_preds_tensor = torch.cat(all_preds, dim=0)
all_gts_tensor = torch.cat(all_gts, dim=0)

print("\nComputing Distributional Metrics (FID/KID)... Cultivating Inception features...")
fid_value = compute_fid(all_preds_tensor, all_gts_tensor, device=device)
kid_mean, kid_std = compute_kid(all_preds_tensor, all_gts_tensor, device=device)

# 4. Final Results Summary
print(f"\n{'='*60}")
print(f"VAL EVALUATION RESULTS (N={len(ssim_scores)})")
print(f"{'='*60}")
print(f"{'Metric':<15} | {'Mean':<12} | {'Std':<12}")
print(f"{'='*15}-|-{'='*12}-|-{'='*12}")
print(f"{'SSIM ↑':<15} | {np.mean(ssim_scores):<12.4f} | {np.std(ssim_scores):<12.4f}")
print(f"{'PSNR ↑':<15} | {np.mean(psnr_scores):<12.4f} | {np.std(psnr_scores):<12.4f}")
print(f"{'LPIPS ↓':<15} | {np.mean(lpips_scores):<12.4f} | {np.std(lpips_scores):<12.4f}")
print(f"{'MAD ↓':<15} | {np.mean(mad_scores):<12.4f} | {np.std(mad_scores):<12.4f}")
print(f"{'FID ↓':<15} | {fid_value:<12.4f} | {'N/A':<12}")
print(f"{'KID ↓':<15} | {kid_mean:<12.4f} | {kid_std:<12.4f}")
print(f"{'='*60}")

## 9. Report Final Metrics to W&B

Upload the calculated evaluation metrics to the current W&B run for tracking and comparison.

In [ ]:
import wandb
import numpy as np

# 1. Prepare metrics dictionary
final_val_metrics = {
    "val/final_ssim": np.mean(ssim_scores),
    "val/final_psnr": np.mean(psnr_scores),
    "val/final_lpips": np.mean(lpips_scores),
    "val/final_mad": np.mean(mad_scores),
    "val/final_fid": fid_value,
    "val/final_kid": kid_mean
}

# 2. Log to active W&B run
if wandb.run is not None:
    wandb.log(final_val_metrics)
    print("\u2713 Successfully reported final metrics to WandB summary:")
    for k, v in final_val_metrics.items():
        print(f"  - {k}: {v:.4f}")
else:
    print("\u26a0\ufe0f No active WandB run found. Metrics printed above but not logged to the cloud.")

## 10. 512×512 Finetuning from 256 Checkpoint

Rescale the trained 256×256 StarDiff model to 512×512 resolution:
- **Bicubic interpolation** of sincos positional embeddings (256 → 1024 tokens)
- **RoPE frequency rescaling** via `pt_seq_len/ft_seq_len` ratio
- Train on preprocessed 512 patches at **native resolution** (no resize)

All pretrained StarDiff weights are preserved — only pos_embed and RoPE are adapted.

⚠️ **512 uses 4× tokens (1024 vs 256) → ~16× attention cost.** Recommended GPU:
| GPU | batch_size | grad_accum | effective |
|-----|-----------|------------|-----------|
| A100-80GB | 2 | 4 | 8 |
| A100-40GB | 1 | 8 | 8 |
| RTX 4090 | 1 | 8 | 8 |

In [ ]:
import os, torch, gc
from stardiff_pixelgen.config import StarDiffPixelGenConfig, detect_gpu
from stardiff_pixelgen.model import create_stardiff_model_for_finetune
from stardiff_pixelgen.dataset import download_dataset, create_dataloaders
from stardiff_pixelgen.train import resume_from_wandb

# ── 1. Configuration for 512 finetuning ──
gpu_type, _, _, precision = detect_gpu()

config_512 = StarDiffPixelGenConfig()
config_512.mixed_precision = precision

# Download checkpoint from W&B
resume_run_id = "8t5yx3xa"  # ← your 256 training run
run_path_512 = f"mohamed-tarek2607/star-diff-he-to-ihc/{resume_run_id}"
checkpoint_file_512 = "stardiff_latest.pt"

download_dir = "./wandb_downloads"
os.makedirs(download_dir, exist_ok=True)
checkpoint_path_256 = os.path.join(download_dir, checkpoint_file_512)

if not os.path.exists(checkpoint_path_256):
    import wandb
    print(f"📥 Downloading {checkpoint_file_512} from W&B run {resume_run_id}...")
    api = wandb.Api()
    run = api.run(run_path_512)
    for f in run.files():
        if checkpoint_file_512 in f.name:
            f.download(root=download_dir, replace=True)
            checkpoint_path_256 = os.path.join(download_dir, f.name)
            print(f"  ✅ Downloaded: {checkpoint_path_256}")
            break
    else:
        raise FileNotFoundError(f"'{checkpoint_file_512}' not found in run {run_path_512}")
else:
    print(f"✓ Checkpoint already downloaded: {checkpoint_path_256}")

# Apply finetune defaults (enables LPIPS=0.1, DINO=0.01, DAB=0.5)
config_512.configure_for_finetune(checkpoint_path_256, target_resolution=512)
config_512.batch_size = 16
config_512.gradient_accumulation_steps = 1

# ── Override dataset/output paths to writable locations ──
config_512.dataset_root = "./data"
config_512.output_dir = "./outputs_512"
os.makedirs(config_512.output_dir, exist_ok=True)

print(f"\nConfig: {config_512.image_size}x{config_512.image_size}, "
      f"batch {config_512.batch_size} x {config_512.gradient_accumulation_steps}")
print(f"Perceptual: LPIPS={config_512.lpips_weight}, DINO={config_512.dino_weight}, "
      f"use_dino={config_512.use_dino}")
print(f"DAB: weight={config_512.dab_weight}, focal={config_512.dab_use_focal}, "
      f"alpha={config_512.dab_focal_alpha}, patches={config_512.dab_patch_sizes}")

# ── 2. Dataset (reuses preprocessed 512 patches at native resolution) ──
download_dataset(config_512.dataset_root, config_512.stains)
train_loader_512, val_loader_512 = create_dataloaders(config_512)

# ── 3. Load trained 256 model → rescale to 512 ──
device = "cuda" if torch.cuda.is_available() else "cpu"
model_512 = create_stardiff_model_for_finetune(
    config=config_512,
    checkpoint_path=checkpoint_path_256,
    new_resolution=512,
    device=device,
)

In [ ]:
gc.collect()
torch.cuda.empty_cache()

from accelerate import Accelerator
from diffusers.optimization import get_constant_schedule_with_warmup
from stardiff_pixelgen.stardiff_scheduler import StarDiffScheduler
from stardiff_pixelgen.losses import PixelGenPerceptualLoss
from stardiff_pixelgen.train import train_stardiff
from src.diffusion.flow_matching.dap_loss import CombinedDABLoss

# ── 4. Scheduler + Perceptual Loss + DAB Loss ──
scheduler_512 = StarDiffScheduler(
    num_timesteps=config_512.num_timesteps,
    restoration_weight=config_512.restoration_weight,
)
perceptual_loss_512 = PixelGenPerceptualLoss(
    use_dino=config_512.use_dino,
    lpips_weight=config_512.lpips_weight,
    dino_weight=config_512.dino_weight,
    noise_gate_threshold=config_512.noise_gate_threshold,
    device=device,
)

# DAB stain-aware loss (patch-level + histogram on deconvolved DAB channel)
dab_loss_512 = None
if config_512.dab_weight > 0:
    dab_loss_512 = CombinedDABLoss(
        patch_sizes=list(config_512.dab_patch_sizes),
        use_focal=config_512.dab_use_focal,
        focal_alpha=config_512.dab_focal_alpha,
        hist_weight=config_512.dab_hist_weight,
        fod_threshold=config_512.dab_fod_threshold,
        weight_alpha=config_512.dab_weight_alpha,
    ).to(device)
    # Freeze (no trainable params, but safety)
    for p in dab_loss_512.parameters():
        p.requires_grad = False
    dab_loss_512.eval()
    print(f"✓ CombinedDABLoss loaded (weight={config_512.dab_weight})")

# ── 5. Optimizer ──
try:
    import bitsandbytes as bnb
    optimizer_512 = bnb.optim.AdamW8bit(
        model_512.parameters(), lr=config_512.learning_rate,
        betas=(0.9, 0.999), weight_decay=1e-2,
    )
    print("✓ Using 8-bit optimizer")
except ImportError:
    from torch.optim import AdamW
    optimizer_512 = AdamW(
        model_512.parameters(), lr=config_512.learning_rate,
        betas=(0.9, 0.999), weight_decay=1e-2,
    )

lr_sched_512 = get_constant_schedule_with_warmup(
    optimizer_512, num_warmup_steps=config_512.warmup_steps,
)

# ── 6. Accelerator ──
accelerator_512 = Accelerator(
    mixed_precision=config_512.mixed_precision,
    gradient_accumulation_steps=config_512.gradient_accumulation_steps,
)
model_512, optimizer_512, train_loader_512, val_loader_512, lr_sched_512 = accelerator_512.prepare(
    model_512, optimizer_512, train_loader_512, val_loader_512, lr_sched_512,
)
print(f"✓ Accelerator ready: {config_512.mixed_precision} precision")

# ── 7. Train at 512 with perceptual + DAB losses! ──
print("=" * 60)
print("Starting 512×512 Finetuning from 256 Checkpoint")
print(f"  Losses: MSE + LPIPS({config_512.lpips_weight}) + DINO({config_512.dino_weight}) + DAB({config_512.dab_weight})")
print("=" * 60)

trained_512 = train_stardiff(
    model=model_512,
    scheduler=scheduler_512,
    train_dataloader=train_loader_512,
    val_dataloader=val_loader_512,
    optimizer=optimizer_512,
    lr_scheduler=lr_sched_512,
    accelerator=accelerator_512,
    config=config_512,
    perceptual_loss_fn=perceptual_loss_512,
    dab_loss_fn=dab_loss_512,
)

## 11. Evaluate 512 Finetuned Model

Run the same evaluation metrics (FID, KID, SSIM, PSNR, MAD, LPIPS) on the 512 finetuned model.

In [ ]:
import torch
import numpy as np
from tqdm import tqdm
from src.eval.metrics import (
    compute_fid, compute_kid, compute_ssim_batch,
    compute_psnr_batch, compute_mad_batch
)
import lpips
import torchvision.transforms.functional as TF

device = "cuda" if torch.cuda.is_available() else "cpu"
lpips_fn = lpips.LPIPS(net="alex").to(device)

model_512.eval()
all_preds_512, all_gts_512 = [], []
ssim_512, psnr_512, mad_512, lpips_512 = [], [], [], []

def denorm(x):
    return ((x + 1) / 2).clamp(0, 1)

print(f"Generating 512×512 predictions for {len(val_loader_512.dataset)} validation samples...")
for batch in tqdm(val_loader_512, desc="512 Evaluation"):
    he = batch["he"].to(device)
    ihc = batch["ihc"].to(device)

    with torch.no_grad():
        unwrapped = accelerator_512.unwrap_model(model_512)
        shape = (he.shape[0], 3, config_512.image_size, config_512.image_size)

        gen = scheduler_512.sample(
            unwrapped, he, shape,
            device=device, use_restoration=True, use_noise=True, progress=False,
        )

        gen_01 = denorm(gen)
        ihc_01 = denorm(ihc)

        ssim_512.extend(compute_ssim_batch(gen_01, ihc_01))
        psnr_512.extend(compute_psnr_batch(gen_01, ihc_01))
        mad_512.extend(compute_mad_batch(gen_01, ihc_01))
        lpips_512.extend(lpips_fn(gen, ihc).view(-1).cpu().tolist())

        all_preds_512.append(gen_01.cpu())
        all_gts_512.append(ihc_01.cpu())

all_preds_t = torch.cat(all_preds_512, dim=0)
all_gts_t = torch.cat(all_gts_512, dim=0)

print("\nComputing FID/KID for 512 model...")
fid_512 = compute_fid(all_preds_t, all_gts_t, device=device)
kid_mean_512, kid_std_512 = compute_kid(all_preds_t, all_gts_t, device=device)

print(f"\n{'='*60}")
print(f"512 FINETUNE EVALUATION (N={len(ssim_512)})")
print(f"{'='*60}")
print(f"{'Metric':<15} | {'Mean':<12} | {'Std':<12}")
print(f"{'='*15}-|-{'='*12}-|-{'='*12}")
print(f"{'SSIM ↑':<15} | {np.mean(ssim_512):<12.4f} | {np.std(ssim_512):<12.4f}")
print(f"{'PSNR ↑':<15} | {np.mean(psnr_512):<12.4f} | {np.std(psnr_512):<12.4f}")
print(f"{'LPIPS ↓':<15} | {np.mean(lpips_512):<12.4f} | {np.std(lpips_512):<12.4f}")
print(f"{'MAD ↓':<15} | {np.mean(mad_512):<12.4f} | {np.std(mad_512):<12.4f}")
print(f"{'FID ↓':<15} | {fid_512:<12.4f} | {'N/A':<12}")
print(f"{'KID ↓':<15} | {kid_mean_512:<12.4f} | {kid_std_512:<12.4f}")
print(f"{'='*60}")

## 12. 1024×1024 Finetuning from 512 Checkpoint

Rescale the 512-finetuned StarDiff model to **native MIST 1024×1024 resolution**:
- **Bicubic interpolation** of sincos positional embeddings (1024 → 4096 tokens)
- **RoPE frequency rescaling** via `pt_seq_len/ft_seq_len` ratio
- Train on **original MIST 1024×1024** paired images (TrainValAB layout)

⚠️ **1024 uses 16× tokens vs 256 (4096 vs 256) → ~256× attention cost.**

| GPU | batch_size | grad_accum | effective |
|-----|-----------|------------|-----------|
| H100-80GB | 1 | 8 | 8 |
| A100-80GB | 1 | 8 | 8 |

**Data**: Original MIST dataset (1024×1024 H&E/IHC pairs) — download from
[Google Drive](https://drive.google.com/drive/folders/146V99Zv1LzoHFYlXvSDhKmflIL-joo6p)
or via Kaggle if uploaded.

In [ ]:
import os, torch, gc
from stardiff_pixelgen.config import StarDiffPixelGenConfig, detect_gpu
from stardiff_pixelgen.model import create_stardiff_model_for_finetune
from stardiff_pixelgen.dataset import download_dataset_1024, create_dataloaders_1024
from stardiff_pixelgen.train import resume_from_wandb

# ── 0. Mode selector ──
# Set to True to continue an already-running/interrupted 1024 training run.
# Set to False to warm-start from a 512 checkpoint (rescales pos_embed).
RESUME_1024_RUN = False

# ── Run IDs ──
resume_run_id_1024 = "REPLACE_WITH_1024_RUN_ID"  # ← used when RESUME_1024_RUN=True
resume_run_id_512  = "REPLACE_WITH_512_RUN_ID"    # ← used when RESUME_1024_RUN=False

# ── 1. Configuration for 1024 finetuning ──
gpu_type, _, _, precision = detect_gpu()

config_1024 = StarDiffPixelGenConfig()
config_1024.mixed_precision = precision

checkpoint_file_1024 = "stardiff_latest.pt"
download_dir = "./wandb_downloads"
os.makedirs(download_dir, exist_ok=True)

# ── 2. Download the right checkpoint ──
if RESUME_1024_RUN:
    # ── Mode A: TRUE CONTINUATION of an existing 1024 run ──
    run_path_for_ckpt = f"mohamed-tarek2607/star-diff-he-to-ihc/{resume_run_id_1024}"
    local_ckpt_name   = f"stardiff_1024_{checkpoint_file_1024}"
else:
    # ── Mode B: WARM-START from a 512 checkpoint ──
    run_path_for_ckpt = f"mohamed-tarek2607/star-diff-he-to-ihc/{resume_run_id_512}"
    local_ckpt_name   = f"stardiff_512_{checkpoint_file_1024}"

checkpoint_path_src = os.path.join(download_dir, local_ckpt_name)

if not os.path.exists(checkpoint_path_src):
    import wandb
    print(f"📥 Downloading {checkpoint_file_1024} from W&B run {run_path_for_ckpt}...")
    api = wandb.Api()
    run = api.run(run_path_for_ckpt)
    for f in run.files():
        if checkpoint_file_1024 in f.name:
            f.download(root=download_dir, replace=True)
            src = os.path.join(download_dir, f.name)
            os.rename(src, checkpoint_path_src)
            print(f"  ✅ Downloaded: {checkpoint_path_src}")
            break
    else:
        raise FileNotFoundError(f"'{checkpoint_file_1024}' not found in run {run_path_for_ckpt}")
else:
    print(f"✓ Checkpoint already downloaded: {checkpoint_path_src}")

# ── 3. Configure ──
config_1024.configure_for_finetune(checkpoint_path_src, target_resolution=1024)
config_1024.batch_size = 8
config_1024.gradient_accumulation_steps = 1
config_1024.dataset_root = "./data"
config_1024.output_dir = "./outputs_1024"
os.makedirs(config_1024.output_dir, exist_ok=True)

print(f"\nMode: {'▶ RESUME 1024 run' if RESUME_1024_RUN else '🔁 WARM-START from 512'}")
print(f"Config: {config_1024.image_size}x{config_1024.image_size}, "
      f"batch {config_1024.batch_size} x {config_1024.gradient_accumulation_steps}")
print(f"Perceptual: LPIPS={config_1024.lpips_weight}, DINO={config_1024.dino_weight}")
print(f"DAB: weight={config_1024.dab_weight}")
print(f"LR: {config_1024.learning_rate}, Epochs: {config_1024.num_epochs}")

# ── 4. Dataset ──
download_dataset_1024(config_1024.dataset_root, config_1024.stains)
train_loader_1024, val_loader_1024 = create_dataloaders_1024(config_1024)

# ── 5. Build model ──
device = "cuda" if torch.cuda.is_available() else "cpu"
# For both modes, create_stardiff_model_for_finetune reads the checkpoint's stored
# image_size and skips rescaling when original_resolution == new_resolution (1024→1024).
model_1024 = create_stardiff_model_for_finetune(
    config=config_1024,
    checkpoint_path=checkpoint_path_src,
    new_resolution=1024,
    device=device,
)

# ── 6. Track start_epoch for Mode A (will be updated after accelerator.prepare) ──
start_epoch_1024 = 0


In [ ]:
gc.collect()
torch.cuda.empty_cache()

from accelerate import Accelerator
from diffusers.optimization import get_constant_schedule_with_warmup
from stardiff_pixelgen.stardiff_scheduler import StarDiffScheduler
from stardiff_pixelgen.losses import PixelGenPerceptualLoss
from stardiff_pixelgen.train import train_stardiff, resume_from_wandb
from src.diffusion.flow_matching.dap_loss import CombinedDABLoss

# ── 4. Scheduler + Perceptual Loss + DAB Loss ──
scheduler_1024 = StarDiffScheduler(
    num_timesteps=config_1024.num_timesteps,
    restoration_weight=config_1024.restoration_weight,
)
perceptual_loss_1024 = PixelGenPerceptualLoss(
    use_dino=config_1024.use_dino,
    lpips_weight=config_1024.lpips_weight,
    dino_weight=config_1024.dino_weight,
    noise_gate_threshold=config_1024.noise_gate_threshold,
    device=device,
)

# DAB stain-aware loss
dab_loss_1024 = None
if config_1024.dab_weight > 0:
    dab_loss_1024 = CombinedDABLoss(
        patch_sizes=list(config_1024.dab_patch_sizes),
        use_focal=config_1024.dab_use_focal,
        focal_alpha=config_1024.dab_focal_alpha,
        hist_weight=config_1024.dab_hist_weight,
        fod_threshold=config_1024.dab_fod_threshold,
        weight_alpha=config_1024.dab_weight_alpha,
    ).to(device)
    for p in dab_loss_1024.parameters():
        p.requires_grad = False
    dab_loss_1024.eval()
    print(f"✓ CombinedDABLoss loaded (weight={config_1024.dab_weight})")

# ── 5. Optimizer (8-bit essential at 1024) ──
try:
    import bitsandbytes as bnb
    optimizer_1024 = bnb.optim.AdamW8bit(
        model_1024.parameters(), lr=config_1024.learning_rate,
        betas=(0.9, 0.999), weight_decay=1e-2,
    )
    print("✓ Using 8-bit optimizer (critical for 1024 VRAM)")
except ImportError:
    from torch.optim import AdamW
    optimizer_1024 = AdamW(
        model_1024.parameters(), lr=config_1024.learning_rate,
        betas=(0.9, 0.999), weight_decay=1e-2,
    )
    print("⚠️ Standard optimizer — may OOM at 1024. Install bitsandbytes!")

lr_sched_1024 = get_constant_schedule_with_warmup(
    optimizer_1024, num_warmup_steps=config_1024.warmup_steps,
)

# ── 6. Accelerator ──
accelerator_1024 = Accelerator(
    mixed_precision=config_1024.mixed_precision,
    gradient_accumulation_steps=config_1024.gradient_accumulation_steps,
)
model_1024, optimizer_1024, train_loader_1024, val_loader_1024, lr_sched_1024 = accelerator_1024.prepare(
    model_1024, optimizer_1024, train_loader_1024, val_loader_1024, lr_sched_1024,
)
print(f"✓ Accelerator ready: {config_1024.mixed_precision} precision")

# ── 7. Full resume (optimizer + scheduler + epoch) for Mode A ──
if RESUME_1024_RUN:
    run_path_resume = f"mohamed-tarek2607/star-diff-he-to-ihc/{resume_run_id_1024}"
    start_epoch_1024 = resume_from_wandb(
        run_path=run_path_resume,
        file_name=checkpoint_file_1024,
        model=model_1024,
        optimizer=optimizer_1024,
        lr_scheduler=lr_sched_1024,
        accelerator=accelerator_1024,
    )
    print(f"✓ Resuming 1024 training from epoch {start_epoch_1024}")
else:
    print("✓ Warm-start from 512 checkpoint — training from epoch 0")

# ── 8. Train at 1024 with all losses! ──
print("=" * 60)
print(f"{'Resuming' if RESUME_1024_RUN else 'Starting'} 1024×1024 Finetuning"
      f"{f' from epoch {start_epoch_1024}' if RESUME_1024_RUN else ' from 512 Checkpoint'}")
print(f"  Losses: MSE + LPIPS({config_1024.lpips_weight}) + DINO({config_1024.dino_weight}) + DAB({config_1024.dab_weight})")
print("=" * 60)

trained_1024 = train_stardiff(
    model=model_1024,
    scheduler=scheduler_1024,
    train_dataloader=train_loader_1024,
    val_dataloader=val_loader_1024,
    optimizer=optimizer_1024,
    lr_scheduler=lr_sched_1024,
    accelerator=accelerator_1024,
    config=config_1024,
    perceptual_loss_fn=perceptual_loss_1024,
    dab_loss_fn=dab_loss_1024,
    start_epoch=start_epoch_1024,
)


## 13. Evaluate 1024 Finetuned Model

Run evaluation metrics (FID, KID, SSIM, PSNR, MAD, LPIPS) on the 1024 model at **native MIST resolution**.

In [ ]:
import torch
import numpy as np
from tqdm import tqdm
from src.eval.metrics import (
    compute_fid, compute_kid, compute_ssim_batch,
    compute_psnr_batch, compute_mad_batch
)
import lpips
import torchvision.transforms.functional as TF

device = "cuda" if torch.cuda.is_available() else "cpu"
lpips_fn = lpips.LPIPS(net="alex").to(device)

model_1024.eval()
all_preds_1024, all_gts_1024 = [], []
ssim_1024, psnr_1024, mad_1024, lpips_1024 = [], [], [], []

def denorm(x):
    return ((x + 1) / 2).clamp(0, 1)

print(f"Generating 1024×1024 predictions for {len(val_loader_1024.dataset)} validation samples...")
for batch in tqdm(val_loader_1024, desc="1024 Evaluation"):
    he = batch["he"].to(device)
    ihc = batch["ihc"].to(device)

    with torch.no_grad():
        unwrapped = accelerator_1024.unwrap_model(model_1024)
        shape = (he.shape[0], 3, config_1024.image_size, config_1024.image_size)

        gen = scheduler_1024.sample(
            unwrapped, he, shape,
            device=device, use_restoration=True, use_noise=True, progress=False,
        )

        gen_01 = denorm(gen)
        ihc_01 = denorm(ihc)

        ssim_1024.extend(compute_ssim_batch(gen_01, ihc_01))
        psnr_1024.extend(compute_psnr_batch(gen_01, ihc_01))
        mad_1024.extend(compute_mad_batch(gen_01, ihc_01))
        lpips_1024.extend(lpips_fn(gen, ihc).view(-1).cpu().tolist())

        all_preds_1024.append(gen_01.cpu())
        all_gts_1024.append(ihc_01.cpu())

all_preds_t = torch.cat(all_preds_1024, dim=0)
all_gts_t = torch.cat(all_gts_1024, dim=0)

print("\nComputing FID/KID for 1024 model...")
fid_1024 = compute_fid(all_preds_t, all_gts_t, device=device)
kid_mean_1024, kid_std_1024 = compute_kid(all_preds_t, all_gts_t, device=device)

print(f"\n{'='*60}")
print(f"1024 FINETUNE EVALUATION (N={len(ssim_1024)})")
print(f"{'='*60}")
print(f"{'Metric':<15} | {'Mean':<12} | {'Std':<12}")
print(f"{'='*15}-|-{'='*12}-|-{'='*12}")
print(f"{'SSIM ↑':<15} | {np.mean(ssim_1024):<12.4f} | {np.std(ssim_1024):<12.4f}")
print(f"{'PSNR ↑':<15} | {np.mean(psnr_1024):<12.4f} | {np.std(psnr_1024):<12.4f}")
print(f"{'LPIPS ↓':<15} | {np.mean(lpips_1024):<12.4f} | {np.std(lpips_1024):<12.4f}")
print(f"{'MAD ↓':<15} | {np.mean(mad_1024):<12.4f} | {np.std(mad_1024):<12.4f}")
print(f"{'FID ↓':<15} | {fid_1024:<12.4f} | {'N/A':<12}")
print(f"{'KID ↓':<15} | {kid_mean_1024:<12.4f} | {kid_std_1024:<12.4f}")
print(f"{'='*60}")